# Aleph Direct-A 10M -> 100M (Staged 10B) Continuation Launcher

Reproducible capped Colab launcher for the frozen **999,352,320-parameter** Aleph Direct-A continuation runner.

- **Never embeds secrets or tokens.** Uses Colab's interactive Google Drive mount.
- **Immutable prior checkpoint:** reads `MyDrive/aleph-v2-run` read-only and never modifies `group_039_7fbdb1f91cf8`.
- **Hard spend/wall caps:** requires explicit `MAX_NEW_GROUPS`, `WALL_SECONDS`, and `CHECKPOINT_RESERVE_SECONDS` before running.
- **Fail-closed verification:** verifies every shard hash on restore and save, publishes `LATEST.json` strictly last, and keeps at least two durable checkpoint generations on Drive.
- **Supports `--device cuda` (A100 40GB) and `--device xla` (TPU v6e-1 when qualified).**

In [ ]:
# === OPERATOR CONFIGURATION (EDIT BEFORE EXECUTING) ===
REPO_URL = "https://github.com/MoralityLabAI/DataGoblin.git"
REPO_BRANCH = "feat/aleph-10b-continuation-prep-20261128"

# Durable Google Drive paths (read-only prior 10M checkpoint, immutable Stage-100M release, new continuation out dir)
DRIVE_PRIOR_10M_DIR = "/content/drive/MyDrive/aleph-v2-run"
PRIOR_MANIFEST_SHA256 = "aca6d5c46301b6bedda375ad3f597dcff29aa4bd7fd1ed0c97677e678a9858ab"
DRIVE_STAGE_RELEASE_DIR = "/content/drive/MyDrive/aleph-10b-continuation-v1/stage_100m"
DRIVE_CONTINUATION_OUT_DIR = "/content/drive/MyDrive/aleph-10b-continuation-v1/runs/stage_100m_a100"

# Local fast NVMe staging paths on the ephemeral Colab runtime
LOCAL_PRIOR_STAGE = "/content/stage_prior_10m"
LOCAL_RELEASE_STAGE = "/content/stage_100m_release"
LOCAL_RUN_OUT = "/content/stage_100m_run"

# Hardware and hard session caps
DEVICE = "cuda"  # "cuda" (A100 40GB) or "xla" (TPU v6e-1 when qualified)
ALLOW_BACKEND_HANDOFF = False  # Set True ONLY when intentionally migrating A100 CUDA -> TPU XLA
MODE = "qualify_1_group"  # "qualify_1_group" | "parity_check" | "continue_capped"
MAX_NEW_GROUPS = 1  # 1 for qualification; up to 344 total for Stage-100M
SAVE_EVERY_GROUPS = 10
WALL_SECONDS = 3300  # e.g. 55 minutes hard cap
CHECKPOINT_RESERVE_SECONDS = 420  # 7 minutes reserved for final save + Drive sync
SYNC_WORKERS = 16
KEEP_GENERATIONS = 2
AUTO_DISCONNECT_ON_FINISH = False  # Set True for unattended capped sessions


In [ ]:
# 1. Mount Google Drive and inspect hardware + durable free storage
import os, shutil, subprocess, sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

if DEVICE == "cuda":
    subprocess.run(["nvidia-smi"], check=True)
elif DEVICE == "xla":
    os.environ["PJRT_DEVICE"] = "TPU"
    print("PJRT_DEVICE=", os.environ["PJRT_DEVICE"])

for label, p in [("local_nvme", "/content"), ("google_drive", "/content/drive/MyDrive")]:
    du = shutil.disk_usage(p)
    print(f"{label}: free={du.free / 1e9:.2f} GB / total={du.total / 1e9:.2f} GB")

assert Path(DRIVE_PRIOR_10M_DIR, "LATEST.json").is_file(), "Missing prior 10M LATEST.json on Drive"
assert Path(DRIVE_STAGE_RELEASE_DIR, "index.json").is_file(), "Missing Stage-100M index.json on Drive"
assert Path(DRIVE_STAGE_RELEASE_DIR, "REPLAY.json").is_file(), "Missing Stage-100M REPLAY.json on Drive"


In [ ]:
# 2. Clone pinned DataGoblin branch and run CPU self-tests before touching checkpoints
repo_dir = Path("/content/DataGoblin")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "checkout", REPO_BRANCH], check=True)

runner_py = repo_dir / "training/aleph_next/continuation10b/runner.py"
sync_py = repo_dir / "training/aleph_next/continuation10b/drive_sync.py"
contract_json = repo_dir / "training/aleph_next/continuation10b/stage_100m.json"

subprocess.run([sys.executable, str(runner_py), "--self-test", "--device", "cpu", "--bf16"], check=True)
subprocess.run([sys.executable, str(runner_py), "--self-test", "--device", DEVICE], check=True)


In [ ]:
# 3. Verify durable Drive read/write + parallel stage corpus & checkpoint onto local NVMe
import json

Path(DRIVE_CONTINUATION_OUT_DIR).mkdir(parents=True, exist_ok=True)
probe_file = Path(DRIVE_CONTINUATION_OUT_DIR) / ".rw_probe.tmp"
probe_file.write_text("ok", encoding="utf-8")
assert probe_file.read_text(encoding="utf-8") == "ok"
probe_file.unlink()
print("Verified durable Drive read/write at:", DRIVE_CONTINUATION_OUT_DIR)

# Copy Stage-100M immutable corpus release to local NVMe
if not Path(LOCAL_RELEASE_STAGE, "index.json").is_file():
    shutil.copytree(DRIVE_STAGE_RELEASE_DIR, LOCAL_RELEASE_STAGE, dirs_exist_ok=True)
print("Staged release locally:", LOCAL_RELEASE_STAGE)

# Determine whether this is a fresh handoff from 10M or a resume of an existing continuation run
is_resume = Path(DRIVE_CONTINUATION_OUT_DIR, "LATEST.json").is_file()
if is_resume:
    print("Resuming existing continuation checkpoint from Drive...")
    subprocess.run([
        sys.executable, str(sync_py), "sync",
        DRIVE_CONTINUATION_OUT_DIR, LOCAL_RUN_OUT,
        "--workers", str(SYNC_WORKERS),
    ], check=True)
else:
    print("Staging immutable prior 10M checkpoint from Drive (read-only)...")
    subprocess.run([
        sys.executable, str(sync_py), "sync",
        DRIVE_PRIOR_10M_DIR, LOCAL_PRIOR_STAGE,
        "--workers", str(SYNC_WORKERS),
    ], check=True)
    latest_prior = json.loads(Path(LOCAL_PRIOR_STAGE, "LATEST.json").read_text(encoding="utf-8"))
    assert latest_prior["sha256"] == PRIOR_MANIFEST_SHA256, "Prior 10M manifest SHA-256 mismatch!"
    print("Verified prior 10M checkpoint SHA-256:", PRIOR_MANIFEST_SHA256)


In [ ]:
# 4. Execute bounded continuation update(s) and sync verified checkpoint back to Drive
cmd = [
    sys.executable, str(runner_py),
    "--execute",
    "--device", DEVICE,
    "--stage-contract", str(contract_json),
    "--release-dir", LOCAL_RELEASE_STAGE,
    "--out", LOCAL_RUN_OUT,
    "--max-new-groups", str(1 if MODE == "qualify_1_group" else MAX_NEW_GROUPS),
    "--save-every-groups", str(SAVE_EVERY_GROUPS),
    "--wall-seconds", str(WALL_SECONDS),
    "--checkpoint-reserve-seconds", str(CHECKPOINT_RESERVE_SECONDS),
]
if is_resume:
    cmd.append("--resume")
else:
    cmd.extend([
        "--handoff-from", LOCAL_PRIOR_STAGE,
        "--prior-manifest-sha256", PRIOR_MANIFEST_SHA256,
    ])
    if ALLOW_BACKEND_HANDOFF:
        cmd.append("--allow-backend-handoff")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

# Sync latest verified checkpoint generation to durable Drive (LATEST.json published strictly last)
subprocess.run([
    sys.executable, str(sync_py), "sync",
    LOCAL_RUN_OUT, DRIVE_CONTINUATION_OUT_DIR,
    "--workers", str(SYNC_WORKERS),
    "--keep-generations", str(KEEP_GENERATIONS),
    "--prune-older",
], check=True)

if AUTO_DISCONNECT_ON_FINISH:
    from google.colab import runtime
    runtime.unassign()
